# Example 3.7 — Market neutral strategy on the SP500 stocks (polars)

A polars rewrite of `python/examples/example3_7.py`.

The strategy shorts the stocks that moved *up* relative to the market index
and buys the ones that moved *down*, so that the market exposure cancels out.
The book reports a Sharpe ratio of about **0.25**, and about **-3.19** once
5 basis points of transaction costs are deducted.

本例是 `python/examples/example3_7.py` 的 polars 版本。

该策略做空那些相对于市场指数上涨的股票、做空下跌的股票，从而抵消市场暴露。
书中给出的夏普比率约为 **0.25**，扣除 5 个基点的交易成本后约为 **-3.19**。


In [ ]:
from pathlib import Path

import numpy as np
import polars as pl
from scipy.io import loadmat


def find_repo_root() -> Path:
    """Walk up from the working directory until the repository root is found.

    从当前工作目录向上查找，直到找到仓库根目录。
    """
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "python" / "epchan").is_dir():
            return base
    raise RuntimeError("could not locate the epchan_book repository root")


REPO_ROOT = find_repo_root()
DATA_FILE = REPO_ROOT / "data" / "SPX_20071123.mat"

# The dates of interest and the one-way transaction cost.
# 关注的日期区间，以及单边交易成本。
START_DATE = 20060101
END_DATE = 20061231
ONE_WAY_TCOST = 0.0005  # assume 5 basis points / 假设为 5 个基点
TRADING_DAYS = 252  # trading days per year / 一年中的交易日数


## 1. Load the price matrix

`SPX_20071123.mat` holds a `tday` vector (2024 dates) and a `cl` matrix of
adjusted close prices (2024 dates x 500 stocks).  polars cannot read MATLAB
files, so `scipy` reads the `.mat` and the arrays are reshaped into a long
("tidy") table: one row per (date, stock) pair.

`SPX_20071123.mat` 中保存了 `tday` 向量（2024 个交易日）和 `cl` 复权收盘价矩阵
（2024 天 x 500 只股票）。polars 无法读取 MATLAB 文件，因此用 `scipy` 读入 `.mat`，
再把数组重塑成长表（一行对应一个「日期-股票」组合）。


In [ ]:
mat = loadmat(DATA_FILE)
tday = mat["tday"].ravel()
cl = mat["cl"]
n_dates, n_symbols = cl.shape

# Long format: one row per (date, stock).  The row-major flattening of ``cl``
# lines up with the repeated dates and the tiled stock indices.
# 转为长表：每行一个（日期，股票）。``cl`` 按行展开后，
# 与重复的日期、铺开的股票编号一一对应。
prices = (
    pl.DataFrame(
        {
            "tday": np.repeat(tday, n_symbols),
            "symbol": np.tile(np.arange(n_symbols), n_dates),
            "close": cl.reshape(-1),
        }
    )
    # MATLAB's NaN becomes a polars null, so that the aggregations below can
    # skip the missing prices the way smartmean/smartsum do in the MATLAB code.
    # MATLAB 的 NaN 转成 polars 的 null，这样下面的聚合就能像 MATLAB 代码里的
    # smartmean/smartsum 那样自动跳过缺失价格。
    .with_columns(pl.col("close").fill_nan(None))
    .sort(["tday", "symbol"])
)

print(f"{n_dates} trading days x {n_symbols} stocks")
prices.head()


## 2. Daily returns and the equal weighted market index

Two quantities are needed per date:

* `ret` — the daily return of every stock, `(cl - lag1(cl)) / lag1(cl)`;
* `mkt_ret` — the equal weighted average of those returns, i.e. the return of
  the market index;
* `n_valid_close` — the number of stocks with a valid price on that date.  This
  is the denominator of the weights below, and it counts *prices*, not returns.

需要为每个日期算出：

* `ret` — 每只股票的日收益率 `(cl - lag1(cl)) / lag1(cl)`；
* `mkt_ret` — 这些收益率的等权平均，即市场指数收益率；
* `n_valid_close` — 当日价格有效的股票数量。它是下面权重的分母，
  统计的是**价格**而不是收益率。


In [ ]:
prices = prices.with_columns(
    pl.col("close").shift(1).over("symbol").alias("close_prev"),
    pl.col("close").pct_change().over("symbol").alias("ret"),
)

market = (
    prices.group_by("tday")
    .agg(
        pl.col("ret").mean().alias("mkt_ret"),
        pl.col("close").is_not_null().sum().alias("n_valid_close"),
    )
    .sort("tday")
)

market.head()


## 3. Weights of the market neutral strategy

The weight of a stock is proportional to the negative distance to the market
index, divided by the number of valid prices:

```
weight = -(ret - mkt_ret) / n_valid_close
```

Stocks without a valid price (or without a valid lagged price, which is the
case on the very first date) get a weight of zero.

市场中性策略的个股权重与其到市场指数的负距离成正比，再除以有效价格的数量：

```
weight = -(ret - mkt_ret) / n_valid_close
```

价格无效（或滞后价格无效，即第一个交易日）的股票，权重置零。


In [ ]:
prices = (
    prices.join(market, on="tday", how="left")
    .with_columns(
        pl.when(pl.col("close").is_null() | pl.col("close_prev").is_null())
        .then(0.0)
        .otherwise(
            -(pl.col("ret") - pl.col("mkt_ret")) / pl.col("n_valid_close")
        )
        .alias("weight")
    )
)

prices.head()


## 4. Daily PnL and Sharpe ratio

The portfolio earns yesterday's weight times today's return, summed over all
stocks.  The Sharpe ratio is the mean daily PnL divided by its standard
deviation, scaled by the square root of the number of trading days.

Note that the MATLAB `smartstd` normalises by **N**, not N-1, so `std(ddof=0)`
is the matching polars call.

组合的每日盈亏等于昨天的权重乘今天的收益率，再对所有股票求和。
夏普比率是每日盈亏的均值除以其标准差，再乘以交易天数的平方根。

注意 MATLAB 的 `smartstd` 按 **N** 归一化而不是 N-1，
因此对应的 polars 写法是 `std(ddof=0)`。


In [ ]:
# A sum over an all-empty group must stay empty, exactly like MATLAB's
# smartsum, which returns NaN when no finite element is left.
# 全为空的分组求和后必须仍为空，这与 MATLAB 的 smartsum 行为一致：
# 当没有有限元素时它返回 NaN。
pnl = (
    prices.with_columns(pl.col("weight").shift(1).over("symbol").alias("weight_lag"))
    .with_columns((pl.col("weight_lag") * pl.col("ret")).alias("contribution"))
    .group_by("tday")
    .agg(
        pl.col("contribution").sum().alias("pnl"),
        pl.col("contribution").count().alias("n_valid"),
    )
    .with_columns(
        pl.when(pl.col("n_valid") == 0)
        .then(None)
        .otherwise(pl.col("pnl"))
        .alias("pnl")
    )
    .sort("tday")
)

in_range = pnl.filter(pl.col("tday").is_between(START_DATE, END_DATE))
sharpe = (
    np.sqrt(TRADING_DAYS)
    * in_range["pnl"].mean()
    / in_range["pnl"].std(ddof=0)
)

print(f"Sharpe ratio = {sharpe:.4f}   (the book says about 0.25)")
in_range.head()


## 5. Transaction costs

Trading costs are proportional to the change of the weights.  The lagged
weights are taken *after* the date filter, so the first date of the period
has no lagged weight and is left out of the average, exactly as in the
MATLAB code.

交易成本与权重的变化量成正比。滞后权重是在**日期过滤之后**才计算的，
因此区间内的第一个交易日没有滞后权重，会像 MATLAB 代码那样被排除在均值之外。


In [ ]:
weights_in_range = prices.filter(
    pl.col("tday").is_between(START_DATE, END_DATE)
)

turnover = (
    weights_in_range.with_columns(
        pl.col("weight").shift(1).over("symbol").alias("weight_lag")
    )
    .with_columns(
        (pl.col("weight") - pl.col("weight_lag")).abs().alias("turnover")
    )
    .group_by("tday")
    .agg(
        pl.col("turnover").sum().alias("turnover"),
        pl.col("turnover").count().alias("n_valid"),
    )
    .with_columns(
        pl.when(pl.col("n_valid") == 0)
        .then(None)
        .otherwise(pl.col("turnover"))
        .alias("turnover")
    )
    .sort("tday")
)

pnl_after_cost = (
    in_range.join(turnover, on="tday", how="left")
    .with_columns(
        (pl.col("pnl") - pl.col("turnover") * ONE_WAY_TCOST).alias("pnl_net")
    )
)

sharpe_after_cost = (
    np.sqrt(TRADING_DAYS)
    * pnl_after_cost["pnl_net"].mean()
    / pnl_after_cost["pnl_net"].std(ddof=0)
)

print(
    f"Sharpe ratio with transaction costs = {sharpe_after_cost:.4f}"
    "   (the book says about -3.19)"
)
pnl_after_cost.head()


## Summary

| Quantity | This notebook | numpy version (`python/examples/example3_7.py`) | The book |
| --- | --- | --- | --- |
| Sharpe ratio | 0.2515 | 0.2515 | about 0.25 |
| Sharpe ratio with 5 bps of costs | -3.1886 | -3.1886 | about -3.19 |

Both numbers agree with the numpy implementation to the last decimal digit.

两个结果都与 numpy 版本完全一致。


In [ ]:
print(f"Sharpe ratio                     = {sharpe:.4f}")
print(f"Sharpe ratio with transaction costs = {sharpe_after_cost:.4f}")
